In [1]:

import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
)
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isEmpty,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    setEF
)
from src.utils.util import (
    loadEstaciones,
    loadEstacionSinCTC
)
from src.processor import (
    XPECProcessor,
    XSIVProcessor,
    SitraProcessor,
    
)
from src.utils.topos import getEstacionamientos
from src.utils.util import loadEstacionSinCTC,loadEstaciones

In [2]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import argparse
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex
import yaml
from tqdm.auto import tqdm

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor, xpec_processor
from src.utils import isValidCode, parallelizeFunction, parseDate, rellenarId
from src.api.APIs import (
    getEstadoCirculacionesTecnicas,
    getPlanificacionCirculacionesTecnicas,
    getCirculacionesPlanificadas)

In [3]:
xpec= XPECProcessor()

In [4]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\xPEC\xPEC_20260414033503.xml")

In [5]:
from lxml import etree

def getRegulation(service: str):
    service = etree.fromstring(service)
    service_info = []

    # Identificadores principales
    ntecnico = service.find("identificator").get("code")
    ncomercial = service.find("identificator").get("comercial_code")

    # Recorre cada <regulation>
    for reg in service.find("regulations").iterchildren():
        journey = []

        # --- Periodo ---
        period = reg.find("period")
        p_inicio = period.get("start").split()[0]
        p_fin = period.get("end").split()[0]

        # --- Operador ---
        operador = reg.find("operator")
        op_info = {}
        if operador is not None:
            for c in operador.iterchildren():
                if "code" in c.keys():
                    op_info[c.tag] = c.get("code")
                elif c.tag == "comercial_association":
                    # Extrae ambos códigos (code_1 y code_2) de comercial_association
                    op_info["comercial_association_code_1"] = c.get("code_1")
                    op_info["comercial_association_code_2"] = c.get("code_2")
                else:
                    op_info[c.tag] = c.get("code_1")
        #----- Rolling_stocks------
        rolling = reg.find("rolling_stock")
        rolling_info = {}

        if rolling is not None:
            for r in rolling.iterchildren():
                for k, v in r.attrib.items():
                    rolling_info[f"{r.tag}_{k}"] = v
                
        # --- Días regulares ---
        reg_days = reg.find("regulation/calendar/regular_days")
        no_dates_node = reg.find("regulation/calendar/no_dates")
        no_dates = []
        if no_dates_node is not None:
            no_dates = [
                d.get("value")
                for d in no_dates_node.findall("date")
                if d.get("value")
            ]

        # --- 🔹 Rotaciones dentro de regulation ---
        rotations_info = []
        rotations = reg.find("regulation/rotations")
        
        if rotations is not None:
            for rot in rotations.iterfind("serviceRotation"):
                rot_info = {}
                identificador = rot.find("identificador")
                if identificador is not None:
                    rot_info["rotation_code"] = identificador.get("code")

                calendar = rot.find("calendar")
                if calendar is not None:
                    rot_info["rotation_calendar_code"] = calendar.get("code")
                    regular_days = calendar.find("regular_days")
                    if regular_days is not None and regular_days.get("value"):
                        rot_info["rotation_regular_days"] = f'{regular_days.get("value"):0>7}'
                    # if no_dates is not None and no_dates.iterfind("date"):
                    #     dates = [d.get("value") for d in no_dates.iterfind("date") if d.get("value")]
                    holiday_region = calendar.find("holidayRegion")
                    if holiday_region is not None:
                        rot_info["rotation_holiday_region"] = holiday_region.get("regionCodeType", "")
                rotations_info.append(rot_info)

        # --- Journey ---
        journey_xml = reg.find("journey")
        if journey_xml is None:
            continue

        for cp in journey_xml.iterchildren():
            has_rolling_inside_cp = cp.find("rolling_stock") is not None
            control_point = {
                "NTécnico": ntecnico,
                "NComercial": ncomercial,
                "rolling_inside_control_point": has_rolling_inside_cp,
                "regular_days": f'{reg_days.get("value"):0>7}' if reg_days is not None else "",
                "no_dates": no_dates,
                "periodo_inicio": p_inicio,
                "periodo_fin": p_fin,
                **op_info,
                **rolling_info                
            }

            # Añade los atributos del punto
            control_point.update(dict(cp.items()))

            # Añade subelementos del punto (ej. <times .../>, <order .../>, <comercial .../>)
            for el in cp.iterchildren():
                # Guarda los atributos de cada subelemento
                for k, v in el.items():
                    control_point[f"{el.tag}_{k}"] = v
                if el.tag == "times":
                    control_point["departure"] = el.get("departure")
                    control_point["technical_departure"] = el.get("technical_departure")

                # ✅ Si el elemento es <comercial>, guarda su "value" aparte
                if el.tag == "comercial" and el.get("value"):
                    control_point["comercial_value"] = el.get("value")

            # 🔸 Combinar con la rotación (si hay)
            if rotations_info:
                for rot in rotations_info:
                    cp_full = control_point.copy()
                    cp_full.update(rot)
                    journey.append(cp_full)
            else:
                journey.append(control_point)

        # Agrega todo al resultado
        service_info.extend(journey)

    return service_info


In [6]:
def readLogFile(fname: Path):
    service_info = []
    tree = etree.parse(fname)
    root = tree.getroot()
    servicios = parallelizeFunction(
        getRegulation, [etree.tostring(s) for s in root.iterchildren()]
    )
    # for service in root.iterchildren():
    #     ntecnico = service.find("identificator").get("code")
    #     ncomercial = service.find("identificator").get("comercial_code")
    #     for reg in service.find("regulations").iterchildren():
    #         journey = []
    #         period = reg.find("period")
    #         operador = reg.find("operator")
    #         op_comercial = reg.find("operator/comercial_association").get("code_1")
    #         reg_days = reg.find("regulation/calendar/regular_days")
    #         # if reg_days is not None:
    #         #     control_point["regular_days"] = reg_days.get("value")
    #         # else:
    #         #     control_point["regular_days"] = None
    #         for cp in reg.find("journey").iterchildren():
    #             control_point = {
    #                 "NTécnico": ntecnico,
    #                 "NComercial": ncomercial,
    #                 "regular_days": f'{reg_days.get("value"):0>7}',
    #                 "periodo_inicio": period.get("start").split()[0],
    #                 "periodo_fin": period.get("end").split()[0],
    #                 "op_comercial": op_comercial,
    #             }
    #             control_point.update(dict(cp.items()))
    #             for el in cp.iterchildren():
    #                 control_point.update(
    #                     {f"{el.tag}_{k}": v for k, v in el.items()}
    #                 )
    #             # if not control_point.get("type"):
    #             #     continue
    #             journey.append(control_point)
    #         service_info.extend(journey)
    for s in servicios:
        service_info.extend(s)
    return service_info

In [7]:
df= readLogFile(fname)

  0%|          | 0/11982 [00:00<?, ?it/s]

In [8]:
df_df = pd.DataFrame(df)

In [9]:
df_df.columns

Index(['NTécnico', 'NComercial', 'rolling_inside_control_point',
       'regular_days', 'no_dates', 'periodo_inicio', 'periodo_fin', 'company',
       'product', 'comercial_product', 'comercial_association_code_1',
       'comercial_association_code_2', 'traction_provider', 'traction_number',
       'traction_code', 'weigth_value', 'length_value', 'security_rule_value',
       'danger_cargo_value', 'tle_value', 'distance_to_previous', 'type',
       'code', 'lineCode', 'lineCode_Complement', 'times_departure',
       'times_technical_departure', 'departure', 'technical_departure',
       'comercial_value', 'order_value', 'times_arrival', 'speed_value',
       'speed_type', 'train_identificator_code', 'class_stop', 'rotation_code',
       'rotation_calendar_code', 'rotation_regular_days',
       'rotation_holiday_region'],
      dtype='object')

<h1>Rotación </h1>

In [10]:
df_df['periodo_inicio'] = pd.to_datetime(df_df['periodo_inicio'])
df_df['periodo_fin'] = pd.to_datetime(df_df['periodo_fin'])

In [11]:
fecha_objetivo = pd.to_datetime("2026-04-14")


df_df = df_df[
    (df_df['periodo_inicio'].dt.date <= fecha_objetivo.date()) &
    (df_df['periodo_fin'].dt.date >= fecha_objetivo.date())]


In [12]:
df_df = df_df[~df_df["no_dates"].apply(lambda x: "2026-04-09" in x if isinstance(x, list) else False)]

In [13]:
rotations = df_df[["NTécnico","regular_days","product","comercial_product","comercial_association_code_1","comercial_association_code_2","rotation_code",
                     "rotation_calendar_code","rotation_regular_days"]].copy()

In [14]:
circulación = rotations[rotations["regular_days"].str[1] == "1"].copy()

In [15]:

circulación["rotación"] = np.where(
    circulación["rotation_regular_days"].str[1] == "1",
   circulación["rotation_code"],
    None
)


In [16]:
test= circulación[(circulación["rotation_regular_days"].str[1]=="1") | (circulación["rotation_regular_days"].isna())].copy()

In [17]:
ambito = pd.read_csv("data/Ámbitos.csv",sep=";")

In [18]:

test["NTécnico"] = test["NTécnico"].astype(int)

In [19]:
ambito["RI"] = ambito["RI"].astype(int)
ambito["RF"] = ambito["RF"].astype(int)

In [20]:
merged = test.merge(ambito, how="cross")


In [21]:

merged = merged[
    (merged["NTécnico"] >= merged["RI"]) &
    (merged["NTécnico"] <= merged["RF"])
].copy()


In [22]:
merged.drop(columns = ["RI","RF"],inplace= True)

In [23]:
merged.reset_index(drop=True,inplace=True)

In [24]:
merged["NTécnico"] = merged["NTécnico"].astype(str)

In [25]:
merged

,NTécnico,regular_days,product,comercial_product,comercial_association_code_1,comercial_association_code_2,rotation_code,rotation_calendar_code,rotation_regular_days,rotación,Ámbito,Subdirección,AM
0,40,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
1,40,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
2,40,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
3,40,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
4,40,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
122168,82891,1111000,M,1,1RM,,NaN,NaN,NaN,None,Otro,NaN,NaN
122169,82891,1111000,M,1,1RM,,NaN,NaN,NaN,None,Otro,NaN,NaN
122170,82891,1111000,M,1,1RM,,NaN,NaN,NaN,None,Otro,NaN,NaN
122171,82891,1111000,M,1,1RM,,NaN,NaN,NaN,None,Otro,NaN,NaN


In [26]:
merged["NTécnico"] = merged["NTécnico"].apply(
    lambda x: rellenarId(x) if isValidCode(x) else x
)

In [27]:
merged

,NTécnico,regular_days,product,comercial_product,comercial_association_code_1,comercial_association_code_2,rotation_code,rotation_calendar_code,rotation_regular_days,rotación,Ámbito,Subdirección,AM
0,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
1,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
2,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
3,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
4,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None,LD,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
122168,82891,1111000,M,1,1RM,,NaN,NaN,NaN,None,Otro,NaN,NaN
122169,82891,1111000,M,1,1RM,,NaN,NaN,NaN,None,Otro,NaN,NaN
122170,82891,1111000,M,1,1RM,,NaN,NaN,NaN,None,Otro,NaN,NaN
122171,82891,1111000,M,1,1RM,,NaN,NaN,NaN,None,Otro,NaN,NaN


In [28]:
merged.drop(columns=["regular_days","product","comercial_product","comercial_association_code_1","comercial_association_code_2","rotation_code","rotation_calendar_code","rotation_regular_days"],inplace=True)

In [29]:
fname= Path(r"c:\Users\xiangzhou.zhang\Documents\Data\xPEC\rotaciones.txt")

In [30]:
rotacion_txt = pd.read_csv(fname,sep="\t", header=None, names=["Fecha", "N1", "N2"],dtype=str )

In [31]:
Fecha = rotacion_txt[rotacion_txt["Fecha"] == "20260325"].copy()

In [32]:
Fecha

,Fecha,N1,N2
0,20260325,37000,75601
1,20260325,75601,75001
2,20260325,75001,75006
3,20260325,75006,75608
4,20260325,75608,75615
...,...,...,...
5287,20260325,87016,NaN
5288,20260325,87336,87422
5289,20260325,87422,NaN
5290,20260325,87401,NaN


In [33]:
merged.drop_duplicates(subset=["NTécnico", "rotación"], inplace= True)

In [34]:
merged.reset_index(drop=True, inplace=True)

In [35]:
resultado = merged.merge(Fecha[["N1", "N2"]], how="left", left_on="NTécnico", right_on="N1")

In [36]:


resultado["comparación"] = resultado.apply(
    lambda x: (
        "NO CIRCULACION" if pd.isna(x["N1"]) else
        ("NO TXT" if pd.notna(x["rotación"]) and pd.isna(x["N2"]) else
         ("NO XPEC" if pd.isna(x["rotación"]) and pd.notna(x["N2"]) else
          ("NO ROTACIÓN" if pd.isna(x["rotación"]) and pd.isna(x["N2"]) else
           (x["rotación"] == x["N2"]))))
    ),
    axis=1
)

In [37]:
circulación_planificadas= getCirculacionesPlanificadas("2026-04-14")


Error: '500' en la respuesta


AttributeError: 'NoneType' object has no attribute 'text'

In [ ]:
cir=circulación[circulación["NTécnico"].isin(circulación_planificadas["NTécnico"])].copy()

In [ ]:
test=cir.drop_duplicates(subset=["NTécnico","rotación"]).copy()

In [ ]:
test1 = test[["NTécnico","rotación"]].copy()

In [ ]:
test1[~test1["rotación"].isna()]

,NTécnico,rotación
24567,05890,05951
34301,08078,08589
47689,13300,23156
47692,13301,13300
47695,13302,23166
...,...,...
316755,79016,79015
316761,79018,79017
316768,79020,79019
316777,79022,79003


In [ ]:
Gct = test1.copy()

In [ ]:
Gct.rename(columns={"NTécnico":"Número técnico tren anterior","rotación": "Número técnico"},inplace=True)

In [ ]:
Gct.reset_index(drop=True,inplace=True)

In [ ]:
circulación_planificadas

,NTécnico,FechaOrigen,Secuencia,Código,Vía_Planificada,Línea,Compañia,Operador,TipoTren
0,15022,2026-04-14,1,71400,3,None,RF,R,REGIONAL EXPRES
1,15022,2026-04-14,2,B7510,2,None,RF,R,REGIONAL EXPRES
2,15022,2026-04-14,3,71401,2,None,RF,R,REGIONAL EXPRES
3,15022,2026-04-14,4,A7141,2,None,RF,R,REGIONAL EXPRES
4,15022,2026-04-14,5,71500,2,None,RF,R,REGIONAL EXPRES
...,...,...,...,...,...,...,...,...,...
145396,13027,2026-04-14,19,50521,1,None,RF,R,MD
145397,13027,2026-04-14,20,50502,1,None,RF,R,MD
145398,13027,2026-04-14,21,50501,1,None,RF,R,MD
145399,13027,2026-04-14,22,B5052,1,None,RF,R,MD


In [ ]:
def obtener_tipo(fila, B):
    id_actual = fila['Número técnico']
    id_anterior = fila['Número técnico tren anterior']

    # Caso sin anterior
    if pd.isna(id_actual):
        return 'no rotación'

    # Filtrar secuencias del bus actual y del anterior
    actual = B[B['NTécnico'] == id_actual].sort_values('Secuencia')
    anterior = B[B['NTécnico'] == id_anterior].sort_values('Secuencia')

    # Verificar que existan secuencias suficientes
    if actual.empty or len(anterior) < 2:
        return 'enlace'

    # Extraer valores clave
    estacion_seq2 = actual.loc[actual['Secuencia'] == 2, 'Código']
    estacion_penultima = anterior.iloc[-2]['Código']  # penúltima

    if estacion_seq2.empty:
        return 'enlace'

    # Comparar
    if estacion_seq2.iloc[0] == estacion_penultima:
        return 'rotación'
    else:
        return 'enlace'

# Aplicar la función
Gct['RotationType'] = Gct.apply(lambda fila: obtener_tipo(fila, circulación_planificadas), axis=1)


In [ ]:
Gct

,Número técnico tren anterior,Número técnico,RotationType
0,00040,None,no rotación
1,00041,None,no rotación
2,00081,None,no rotación
3,00083,None,no rotación
4,00190,None,no rotación
...,...,...,...
5357,82310,None,no rotación
5358,82502,None,no rotación
5359,82510,None,no rotación
5360,82522,None,no rotación


In [ ]:
"""
Información sobre circulacion de un día concreto
"""
from datetime import date
from src.api.APIs import hacerPeticion, parse_launching_date


HOSTPATH = "http://info.api.elcano.operaciones.adif/mse-circulations/msecirculations/planning/day/"
data = {"day": "2026-04-14"}
data = json.dumps(data)
response = hacerPeticion(
    "POST",
    HOSTPATH,
    data=data,  
)
res_data = regex.sub(r"\n*data:\s*", ",", response.text)[1:]
res_data = json.loads(f"[{res_data}]")
rows = []
for el in res_data:
    cid = el.get("circulationId", {}) or {}
    tecnico = cid.get("number")
    fecha = parse_launching_date(cid.get("launchingDate"))
    day_train = el.get("dayTrain", {}) or {}
    line_dict = day_train.get("line") or {}  
    line = line_dict.get("name")  
    company = day_train.get("company")
    operator = day_train.get("operator")
    train_type = day_train.get("trainType")
    steps = day_train.get("journey", {}).get("steps") or []
    for s in steps:
        rows.append({
            "NTécnico": tecnico,
            "FechaOrigen": fecha,
            "Secuencia": s.get("step"),
            "Código": s.get("pointId"),   
            "Vía_Planificada": s.get("parkingTrack"),
            "Línea": line,
            "Compañia": company,
            "Operador": operator,
            "TipoTren": train_type,
            "Llegada":s.get("arrive"),
            "Salida":s.get("leave")
        })

planificacion = pd.DataFrame(rows)


In [ ]:
planificacion['Llegada'] = pd.to_datetime(planificacion['Llegada'], unit='ms')

In [ ]:
planificacion['Salida'] = pd.to_datetime(planificacion['Salida'], unit='ms')

In [ ]:
planif  = planificacion[["NTécnico","FechaOrigen","Código","Secuencia","Llegada","Salida"]].copy()

In [ ]:
planif['Secuencia'] = planif['Secuencia'].astype(int)


In [ ]:
planif_filtrado = planif.sort_values(['NTécnico', 'Secuencia']).groupby('NTécnico').agg({
    'FechaOrigen': 'first',
    'Código': ['first', 'last'],
    'Secuencia': ['first', 'last'],
    'Llegada': ['first', 'last'],
    'Salida': ['first', 'last']
}).reset_index()


In [ ]:
planif_filtrado.columns = ['_'.join(col).strip('_') for col in planif_filtrado.columns.values]

In [ ]:
planif_filtrado

,NTécnico,FechaOrigen_first,Código_first,Código_last,Secuencia_first,Secuencia_last,Llegada_first,Llegada_last,Salida_first,Salida_last
0,00040,2026-04-14,71801,51003,1,110,2026-04-14 06:30:00,2026-04-14 12:47:00,2026-04-14 06:30:00,2026-04-14 12:47:00
1,00041,2026-04-14,51003,71801,1,110,2026-04-14 14:17:00,2026-04-14 20:59:50,2026-04-14 14:17:00,2026-04-14 20:59:50
2,00081,2026-04-14,51003,03216,1,68,2026-04-14 14:44:00,2026-04-14 19:06:30,2026-04-14 14:44:00,2026-04-14 19:06:30
3,00083,2026-04-14,03216,51003,1,68,2026-04-14 07:15:00,2026-04-14 11:35:50,2026-04-14 07:15:00,2026-04-14 11:35:50
4,00190,2026-04-14,17000,37606,1,55,2026-04-14 06:30:00,2026-04-14 11:26:00,2026-04-14 06:30:00,2026-04-14 11:26:00
...,...,...,...,...,...,...,...,...,...,...
6518,97320,2026-04-14,71801,A0660,1,3,2026-04-14 10:45:00,2026-04-14 10:51:00,2026-04-14 10:45:00,2026-04-14 10:51:00
6519,99009,2026-04-14,37610,37606,1,3,2026-04-14 03:00:00,2026-04-14 03:07:02,2026-04-14 03:00:00,2026-04-14 03:07:02
6520,99017,2026-04-14,33016,C3308,1,2,2026-04-14 03:00:00,2026-04-14 03:03:00,2026-04-14 03:00:00,2026-04-14 03:03:00
6521,99019,2026-04-14,71121,C3308,1,117,2026-04-14 02:00:00,2026-04-14 12:58:00,2026-04-14 02:00:00,2026-04-14 12:58:00


In [ ]:

Gct_merged = Gct.merge(
    planif_filtrado[['NTécnico', 'Llegada_last']],
    left_on='Número técnico tren anterior',
    right_on='NTécnico',
    how='left'
).rename(columns={'Llegada_last': 'Llegada_last_anterior'}) \
 .drop(columns='NTécnico')

Gct_merged = Gct_merged.merge(
    planif_filtrado[['NTécnico', 'Salida_first']],
    left_on='Número técnico',
    right_on='NTécnico',
    how='left'
).rename(columns={'Salida_first': 'Salida_first_actual'}) \
 .drop(columns='NTécnico')

# --- convertir a datetime si hace falta ---
Gct_merged['Llegada_last_anterior'] = pd.to_datetime(Gct_merged['Llegada_last_anterior'])
Gct_merged['Salida_first_actual'] = pd.to_datetime(Gct_merged['Salida_first_actual'])

# --- calcular solo para las filas con rotación ---
mask = Gct_merged['RotationType'] == 'rotación'
Gct_merged.loc[mask, 'planned_time_to_rotate'] = (
    Gct_merged['Salida_first_actual'] - Gct_merged['Llegada_last_anterior']
)

# --- opcional: también en minutos ---
Gct_merged['planned_time_to_rotate_min'] = (
    Gct_merged['planned_time_to_rotate'].dt.total_seconds() / 60
)


In [ ]:
Gct_merged["Número técnico"] = Gct_merged["Número técnico"].astype("Int64")
merged_1 = Gct_merged.merge(ambito, how="cross")


In [ ]:
merged_1 = merged_1[
    (merged_1["Número técnico"] >= merged_1["RI"]) &
    (merged_1["Número técnico"] <= merged_1["RF"])
].copy()


In [ ]:
merged_1.drop(columns=["RI", "RF"],inplace=True)


In [ ]:
centro = merged_1[merged_1["Subdirección"] == "CENTRO"].copy()

In [ ]:
centro["MinimumTimeToRotate"] = np.where(
    centro["RotationType"] == "rotación", 3,
    np.where(centro["RotationType"] == "enlace", 1, np.nan)  
)

In [ ]:
centro = centro[["Ámbito","Subdirección","AM","Número técnico tren anterior","Número técnico","RotationType","Llegada_last_anterior","Salida_first_actual","planned_time_to_rotate","planned_time_to_rotate_min","MinimumTimeToRotate"]]

In [ ]:
merged.drop(columns="AM", inplace=True)

In [ ]:
Fecha_1 = Fecha.merge(ambito, how="cross")


In [ ]:
Fecha_1

,Fecha,N1,N2,RI,RF,Ámbito,Subdirección,AM
0,20260325,37000,75601,0,999,LD,NaN,NaN
1,20260325,37000,75601,1000,1999,LD,NaN,NaN
2,20260325,37000,75601,2000,5999,AV,NaN,NaN
3,20260325,37000,75601,6000,7999,AV,NaN,NaN
4,20260325,37000,75601,8000,9999,AV,NaN,NaN
...,...,...,...,...,...,...,...,...
317515,20260325,67266,NaN,76000,77999,Cercanías,NaN,NaN
317516,20260325,67266,NaN,78000,79999,Otro,NaN,NaN
317517,20260325,67266,NaN,80000,89999,Otro,NaN,NaN
317518,20260325,67266,NaN,90000,97999,Otro,NaN,NaN


In [ ]:
Fecha_1["N1"] = Fecha_1["N1"].astype(int)
Fecha_2 = Fecha_1[
    (Fecha_1["N1"] >= Fecha_1["RI"]) &
    (Fecha_1["N1"] <= Fecha_1["RF"])
].copy()
Fecha_2.drop(columns = ["RI","RF"],inplace= True)
Fecha_2.reset_index(drop=True,inplace=True)


In [ ]:
Fecha_2["N1"] = Fecha_2["N1"].astype(str)

In [ ]:
Fecha_2.drop(columns= "AM", inplace=True)

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\rotación_1.xlsx")

In [ ]:
data= {
    "xpec":merged,
    # "GCT": centro,
    "txt":Fecha_2,
    "Comparación": resultado,

}

In [ ]:
guardarExcelMulti(data,fname)

Guardando: xpec (1/3)
Guardando: txt (2/3)
Guardando: Comparación (3/3)


<h1> Tren Origen </>

In [ ]:
rotaciones = merged_1[merged_1["RotationType"] == "rotación"].copy()

In [ ]:
merged_1

,Número técnico tren anterior,Número técnico,RotationType,Llegada_last_anterior,Salida_first_actual,planned_time_to_rotate,planned_time_to_rotate_min,Ámbito,Subdirección,AM
13682,05890,5951,rotación,2026-04-14 10:49:00,2026-04-14 12:11:00,0 days 01:22:00,82.0,AV,NaN,NaN
18904,08078,8589,rotación,2026-04-14 06:28:00,2026-04-14 16:15:00,0 days 09:47:00,587.0,AV,NaN,NaN
34990,13300,23156,enlace,2026-04-14 08:54:00,2026-04-14 08:55:00,NaT,NaN,Cercanías,SUR,NaN
35046,13301,13300,rotación,2026-04-14 08:37:00,2026-04-14 08:43:00,0 days 00:06:00,6.0,MD,NaN,NaN
35110,13302,23166,enlace,2026-04-14 14:51:00,2026-04-14 14:55:00,NaT,NaN,Cercanías,SUR,NaN
...,...,...,...,...,...,...,...,...,...,...
320636,79016,79015,rotación,2026-04-14 11:46:00,2026-04-14 12:02:00,0 days 00:16:00,16.0,Otro,NaN,NaN
320756,79018,79017,rotación,2026-04-14 13:45:00,2026-04-14 14:07:00,0 days 00:22:00,22.0,Otro,NaN,NaN
320936,79020,79019,rotación,2026-04-14 15:48:00,2026-04-14 16:07:00,0 days 00:19:00,19.0,Otro,NaN,NaN
321116,79022,79003,enlace,2026-04-14 16:39:00,NaT,NaT,NaN,Otro,NaN,NaN


In [ ]:
rotaciones["Número técnico"] = rotaciones["Número técnico"].apply(lambda x: rellenarId(str(x)) if isValidCode(str(x)) else x)

In [ ]:
def obtener_cadena_rotacion(df, ntecnico_inicial):
    cadena = [ntecnico_inicial]
    actual = ntecnico_inicial

    while True:
        fila = df[df["rotación"] == actual]

        if fila.empty:
            # No hay tren anterior → encontramos el origen
            break
        
        # Obtener el tren anterior (NTécnico)
        anterior = fila["NTécnico"].iloc[0]
        cadena.append(anterior)
        actual = anterior

    return cadena[::-1]   # devolver desde el origen al final



In [ ]:
origenes = merged.copy()

In [ ]:
destinos = merged[merged["rotación"].isna()].copy()

In [ ]:
destinos[destinos["Ámbito"] == "Cercanías"]

,NTécnico,rotación,Ámbito,Subdirección
1047,19647,None,Cercanías,CENTRO
1109,19761,None,Cercanías,CENTRO
1121,19775,None,Cercanías,CENTRO
1129,19785,None,Cercanías,CENTRO
1132,19789,None,Cercanías,CENTRO
...,...,...,...,...
5000,77664,None,Cercanías,NaN
5022,77762,None,Cercanías,NaN
5024,77766,None,Cercanías,NaN
5025,77768,None,Cercanías,NaN


In [ ]:

rotacion_a_tecnico = dict(zip(origenes["rotación"], origenes["NTécnico"]))

def obtener_origen(ntecnico):
    actual = ntecnico
    while actual in rotacion_a_tecnico:
        actual = rotacion_a_tecnico[actual]
    return actual

# Asignar origen a cada fila
destinos["origen"] = destinos["NTécnico"].apply(obtener_origen)


In [ ]:
destinos

,NTécnico,rotación,Ámbito,Subdirección,origen
0,00040,None,LD,NaN,00040
1,00041,None,LD,NaN,00041
2,00081,None,LD,NaN,00081
3,00083,None,LD,NaN,00083
4,00190,None,LD,NaN,00190
...,...,...,...,...,...
5171,82502,None,Otro,NaN,82502
5172,82510,None,Otro,NaN,82510
5173,82515,None,Otro,NaN,82515
5174,82522,None,Otro,NaN,82522


In [ ]:
primero = df_df[df_df["NTécnico"].isin(destinos["origen"])].copy()

In [ ]:
primero.reset_index(drop=True,inplace=True)

In [ ]:
primero_filtrado = primero[["NTécnico","product","comercial_association_code_1","comercial_association_code_2","comercial_value","code"]].copy()

In [ ]:
origen = primero_filtrado[primero_filtrado['comercial_value'].str.lower() == 'origen']
fin = primero_filtrado[primero_filtrado['comercial_value'].str.lower() == 'destino']

In [ ]:
df_final = pd.merge(
    origen[['NTécnico','comercial_association_code_1', 'comercial_association_code_2','product','code']],
    fin[['NTécnico','comercial_association_code_1', 'comercial_association_code_2', 'code']],
    on=['NTécnico','comercial_association_code_1', 'comercial_association_code_2'],
    suffixes=('_origen', '_final')
)



In [ ]:
df_final = df_final.drop_duplicates()

In [ ]:
df_final

,NTécnico,comercial_association_code_1,comercial_association_code_2,product,code_origen,code_final
0,00040,1L63S14,2L63S14,L,71801,51003
1,00041,1L60D08,2L60D08,L,51003,71801
2,00081,1L64E06,2L64E06,L,51003,03216
3,00083,1L63S14,2L63S14,L,03216,51003
4,00190,1L57T16,2L57T16,L,17000,37606
...,...,...,...,...,...,...
2376,82502,1RM,,M,95104,71902
2377,82510,1RM,,M,95104,72006
2378,82515,1RM,,M,B3730,71403
2379,82522,1RM,,M,13408,71901


In [ ]:
estaciones = loadEstaciones()
estaciones_sinctc = loadEstacionSinCTC()
estacion = pd.concat([estaciones, estaciones_sinctc], ignore_index=True)

In [ ]:
estaciones

,Delegación,Catálogo,CTC,NombreCTC,Tecnólogo,Código,Nombre,Mnemónico,Mnemónico_comercial
0,SUR,MAL1,MAL,Málaga,DIMETRONIC,54503,Guadalhorce,LG,GDH
1,SUR,MAL1,MAL,Málaga,DIMETRONIC,54412,Los Prados,LG,LG
2,SUR,MAL1,MAL,Málaga,DIMETRONIC,54511,Benalmadena,BD,BD
3,SUR,MAL1,MAL,Málaga,DIMETRONIC,54516,Fuengirola,FE,FE
4,SUR,MAL1,MAL,Málaga,DIMETRONIC,54517,Málaga Centro Alameda,ML,MCA
...,...,...,...,...,...,...,...,...,...
1785,CENTRO,OUR1,ORE,Ourense,ELIOP,31312,Vedra-Rivadulla,VR,VR
1786,CENTRO,OUR1,ORE,Ourense,ELIOP,23006,Portas,XT,XT
1787,CENTRO,OUR1,ORE,Ourense,ELIOP,31205,A Gudiña,AG,AG
1788,CENTRO,OUR1,ORE,Ourense,ELIOP,22100,Ourense,OE,OE


In [ ]:
tren_origen = df_final.merge(
    estacion[['Código', 'Nombre']],
    left_on='code_origen',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre origen',"code_origen":"Código origen"}).drop(columns='Código')

In [ ]:
tren_origen = tren_origen.merge(
    estacion[['Código', 'Nombre']],
    left_on='code_final',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre destino',"code_final":"Código destino"}).drop(columns='Código')

In [ ]:
tren_origen = tren_origen[["NTécnico","comercial_association_code_1","comercial_association_code_2","product","Código origen","Nombre origen","Código destino","Nombre destino"]]

In [ ]:
regex = r'\s+(RAM|AV|RC)$'
tren_origen['Nombre origen'] = tren_origen['Nombre origen'].str.replace(regex, '', regex=True)
tren_origen['Nombre destino'] = tren_origen['Nombre destino'].str.replace(regex, '', regex=True)

In [ ]:
tren_origen = tren_origen.drop_duplicates(subset=["NTécnico"]).copy()  


In [ ]:
ambito = pd.read_csv("data/Ámbitos.csv",sep=";")
tren_origen["NTécnico"] = tren_origen["NTécnico"].astype(int)
ambito["RI"] = ambito["RI"].astype(int)
ambito["RF"] = ambito["RF"].astype(int)
test2 = tren_origen.merge(ambito, how="cross")
test2 = test2[
    (test2["NTécnico"] >= test2["RI"]) &
    (test2["NTécnico"] <= test2["RF"])
].copy()
test2.drop(columns = ["RI","RF"],inplace= True)
test2.reset_index(drop=True,inplace=True)

In [ ]:
test2.drop(columns=["Ámbito","AM"], inplace=True)

In [ ]:
test2 =test2[["Subdirección","NTécnico","comercial_association_code_1","comercial_association_code_2","product","Código origen","Nombre origen","Código destino","Nombre destino"]].copy()

In [ ]:

test2["NTécnico"] = (
    test2["NTécnico"]
    .astype(str)
    .apply(lambda x: rellenarId(x) if isValidCode(x) else x)
)


In [ ]:
tren_origen = test2.copy()

In [ ]:
planifi = getPlanificacionCirculacionesTecnicas("2026-04-14")

In [ ]:
tren_origen_comercial = tren_origen.merge(
    planifi[['NTécnico',"esComercial"]],
    on='NTécnico',
    how='left'
)

In [ ]:
tren_origen_comercial = tren_origen_comercial[tren_origen_comercial["esComercial"] == True].copy()

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\trenes_comienzo_1.xlsx")

In [ ]:
data ={
    "trenes_comienzo": tren_origen,
    "tren_comienzo_comercial": tren_origen_comercial
}

In [ ]:
guardarExcelMulti(data,fname)

Guardando: trenes_comienzo (1/2)
Guardando: tren_comienzo_comercial (2/2)


<h1>Recorrido</h1>

In [ ]:
df_df.drop(columns=["no_dates"],inplace=True)

In [ ]:
no_asociacion = df_df[df_df["comercial_association_code_1"].isna() & df_df["comercial_association_code_2"].isna()].copy()

In [ ]:
no_asociacion2 = df_df[df_df["comercial_association_code_1"].isna() | df_df["comercial_association_code_2"].isna()].copy()

In [ ]:
df_df[df_df["comercial_association_code_2"].isna()]

,NTécnico,NComercial,rolling_inside_control_point,regular_days,periodo_inicio,periodo_fin,company,product,comercial_product,comercial_association_code_1,...,order_value,times_arrival,speed_value,speed_type,train_identificator_code,class_stop,rotation_code,rotation_calendar_code,rotation_regular_days,rotation_holiday_region


In [ ]:
TEST = df_df[(df_df["comercial_association_code_1"] == "1L46T02") & (df_df["comercial_association_code_2"] == "2L46T02")]

In [ ]:
test_origen = TEST[TEST["comercial_value"] == "origen"].copy()

In [ ]:
test_origen[test_origen["code"] == "71801"]

,NTécnico,NComercial,rolling_inside_control_point,regular_days,periodo_inicio,periodo_fin,company,product,comercial_product,comercial_association_code_1,...,order_value,times_arrival,speed_value,speed_type,train_identificator_code,class_stop,rotation_code,rotation_calendar_code,rotation_regular_days,rotation_holiday_region
1988,00530,00530,False,1111111,2026-03-29,2026-12-12,RF,L,V,1L46T02,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2568,00564,00564,False,0000100,2026-03-29,2026-12-12,RF,L,V,1L46T02,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
test_final = TEST[TEST["comercial_value"] == "destino"].copy()

In [ ]:
test_final[test_final["code"] == "71801"]

,NTécnico,NComercial,rolling_inside_control_point,regular_days,periodo_inicio,periodo_fin,company,product,comercial_product,comercial_association_code_1,...,order_value,times_arrival,speed_value,speed_type,train_identificator_code,class_stop,rotation_code,rotation_calendar_code,rotation_regular_days,rotation_holiday_region
2191,00532,00532,False,1111111,2026-03-29,2026-12-12,RF,L,V,1L46T02,...,113,0d 13:09:10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4054,00634,00634,False,0000001,2026-03-29,2026-12-12,RF,L,5,1L46T02,...,80,0d 20:02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40732,10634,10634,False,0000001,2026-04-07,2026-12-12,RF,L,5,1L46T02,...,80,0d 20:02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
lista_subdfs = [group for _, group in df_df.groupby(['NTécnico'])]

In [ ]:
lista_subdfs[0]

,NTécnico,NComercial,rolling_inside_control_point,regular_days,periodo_inicio,periodo_fin,company,product,comercial_product,comercial_association_code_1,...,order_value,times_arrival,speed_value,speed_type,train_identificator_code,class_stop,rotation_code,rotation_calendar_code,rotation_regular_days,rotation_holiday_region
0,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,2,0d 08:35:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,3,0d 08:37:10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,4,0d 08:41:10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,6,0d 08:44:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,142,0d 14:11:40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,143,0d 14:16:20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,144,0d 14:23:10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,146,0d 14:37:50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
lista_viernes = []
for dfs in lista_subdfs:
    viernes = dfs[dfs["regular_days"].str[4] == "1"].copy()
    origen_destino = viernes[~viernes["comercial_value"].isna()].copy()
    lista_viernes.append(origen_destino)

In [ ]:
lista_viernes[0]

,NTécnico,NComercial,rolling_inside_control_point,regular_days,periodo_inicio,periodo_fin,company,product,comercial_product,comercial_association_code_1,...,order_value,times_arrival,speed_value,speed_type,train_identificator_code,class_stop,rotation_code,rotation_calendar_code,rotation_regular_days,rotation_holiday_region
0,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
109,00040,00040,False,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,...,149,0d 14:47:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
recorrido = pd.concat(lista_viernes, ignore_index=True)

In [ ]:
origen = recorrido[recorrido['comercial_value'].str.lower() == 'origen']
fin = recorrido[recorrido['comercial_value'].str.lower() == 'destino']

In [ ]:
df_final = pd.merge(
    origen[['NTécnico','comercial_association_code_1', 'comercial_association_code_2', 'code','product']],
    fin[['NTécnico','comercial_association_code_1', 'comercial_association_code_2', 'code']],
    on=['NTécnico','comercial_association_code_1', 'comercial_association_code_2'],
    suffixes=('_origen', '_final')
)

df_final = df_final.drop_duplicates(subset=['code_origen', 'code_final'])

In [ ]:
df_final

,NTécnico,comercial_association_code_1,comercial_association_code_2,code_origen,product,code_final
0,00040,1L63S14,2L63S14,71801,L,51003
1,00041,1L60D08,2L60D08,51003,L,71801
2,00081,1L64E06,2L64E06,51003,L,03216
3,00083,1L63S14,2L63S14,03216,L,51003
4,00190,1L57T16,2L57T16,17000,L,37606
...,...,...,...,...,...,...
5993,78632,15 1 C8,,79100,C,72209
6030,82001,1RM,,71902,M,95104
6032,82101,1RM,,71901,M,95101
6034,82502,1RM,,95104,M,71902


In [ ]:
mask_b = (
    df_df.groupby("NTécnico")["code"]
     .apply(lambda x: (x == "10101").any())
)

In [ ]:
mask_b

NTécnico
00040    False
00041    False
00081    False
00083    False
00190    False
         ...  
82854    False
82856    False
82891    False
82911     True
83510     True
Name: code, Length: 10110, dtype: bool

In [ ]:
df_final["Paso por Las Matas"] = df_final["NTécnico"].map(mask_b).fillna(False)

In [ ]:
df_final["code_origen"] = df_final["code_origen"].astype(str)
df_final.loc[df_final["code_origen"] == "10101", "Paso por Las Matas"] = "ORIGEN"

In [ ]:
df_final["code_final"] = df_final["code_final"].astype(str)
df_final.loc[df_final["code_final"] == "10101", "Paso por Las Matas"] = "DESTINO"

In [ ]:
df_final["Paso por Las Matas"].unique()

array([False, True], dtype=object)

In [ ]:
df_final

,NTécnico,comercial_association_code_1,comercial_association_code_2,code_origen,product,code_final,Paso por Las Matas
0,00040,1L63S14,2L63S14,71801,L,51003,False
1,00041,1L60D08,2L60D08,51003,L,71801,False
2,00081,1L64E06,2L64E06,51003,L,03216,False
3,00083,1L63S14,2L63S14,03216,L,51003,False
4,00190,1L57T16,2L57T16,17000,L,37606,False
...,...,...,...,...,...,...,...
5993,78632,15 1 C8,,79100,C,72209,False
6030,82001,1RM,,71902,M,95104,True
6032,82101,1RM,,71901,M,95101,False
6034,82502,1RM,,95104,M,71902,True


In [ ]:
df_final.drop(columns=['NTécnico'], inplace=True)

In [ ]:
estaciones_sinctc = loadEstacionSinCTC()

In [ ]:
estacioones = loadEstaciones()

In [ ]:
estaciones = pd.concat([estacioones, estaciones_sinctc], ignore_index=True)

In [ ]:
estaciones = estaciones[["Código","Nombre"]].copy()

In [ ]:
recorrido2 = pd.merge(
    df_final,
    estaciones,
    left_on = "code_origen",
    right_on = "Código",
    how = "left"
)

In [ ]:
recorrido2.drop(columns="Código",inplace=True)

In [ ]:
recorrido2 = pd.merge(
    recorrido2,
    estaciones,
    left_on = "code_final",
    right_on = "Código",
    how = "left"
)

In [ ]:
recorrido2

,comercial_association_code_1,comercial_association_code_2,code_origen,product,code_final,Paso por Las Matas,Nombre_x,Código,Nombre_y
0,1L63S14,2L63S14,71801,L,51003,False,Barcelona-Sants RC,51003,Sevilla-Santa Justa
1,1L63S14,2L63S14,71801,L,51003,False,Barcelona-Sants RC,51003,Sevilla-Santa Justa
2,1L63S14,2L63S14,71801,L,51003,False,Barcelona-Sants AV,51003,Sevilla-Santa Justa
3,1L63S14,2L63S14,71801,L,51003,False,Barcelona-Sants AV,51003,Sevilla-Santa Justa
4,1L60D08,2L60D08,51003,L,71801,False,Sevilla-Santa Justa,71801,Barcelona-Sants RC
...,...,...,...,...,...,...,...,...,...
1589,15 1 C8,,79100,C,72209,False,Granollers-Centre,72209,Martorell
1590,1RM,,71902,M,95104,True,BARCELONA-MORROT,95104,MADRID-ABROÑIGAL
1591,1RM,,71901,M,95101,False,Barcelona Can Tunis,95101,VILLAVERDE-MERCANCIAS
1592,1RM,,95104,M,71902,True,MADRID-ABROÑIGAL,71902,BARCELONA-MORROT


In [ ]:
recorrido2.drop(columns="Código",inplace=True)

In [ ]:
renamed_columns ={
    "comercial_association_code_1":"Código asociación comercial_1",
    "comercial_association_code_2":"Código asociación comercial_2",
    "code_origen":"Código origen",
    "code_final":"Código destino",
    "Nombre_x": "Nombre origen",
    "Nombre_y":"Nombre_destino",
    "product":"Producto"
}

In [ ]:
recorrido2.rename(columns=renamed_columns,inplace=True)

In [ ]:
recorrido2 = recorrido2[["Código asociación comercial_1","Código asociación comercial_2","Producto","Código origen","Código destino","Nombre origen","Nombre_destino","Paso por Las Matas"]].copy()

In [ ]:
regex = r'\s+(RAM|AV|RC)$'


In [ ]:
recorrido2['Nombre origen'] = recorrido2['Nombre origen'].str.replace(regex, '', regex=True)

In [ ]:
recorrido2['Nombre_destino'] = recorrido2['Nombre_destino'].str.replace(regex, '', regex=True)

In [ ]:
recorrido2.drop_duplicates(subset=["Código origen","Código destino"], inplace=True)

In [ ]:
recorrido2

,Código asociación comercial_1,Código asociación comercial_2,Producto,Código origen,Código destino,Nombre origen,Nombre_destino,Paso por Las Matas
0,1L63S14,2L63S14,L,71801,51003,Barcelona-Sants,Sevilla-Santa Justa,False
4,1L60D08,2L60D08,L,51003,71801,Sevilla-Santa Justa,Barcelona-Sants,False
8,1L64E06,2L64E06,L,51003,03216,Sevilla-Santa Justa,Valencia-Joaquin Sorolla,False
12,1L63S14,2L63S14,L,03216,51003,Valencia-Joaquin Sorolla,Sevilla-Santa Justa,False
16,1L57T16,2L57T16,L,17000,37606,Chamartín,Badajoz,False
...,...,...,...,...,...,...,...,...
1589,15 1 C8,,C,79100,72209,Granollers-Centre,Martorell,False
1590,1RM,,M,71902,95104,BARCELONA-MORROT,MADRID-ABROÑIGAL,True
1591,1RM,,M,71901,95101,Barcelona Can Tunis,VILLAVERDE-MERCANCIAS,False
1592,1RM,,M,95104,71902,MADRID-ABROÑIGAL,BARCELONA-MORROT,True


In [ ]:
fname=Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\recorrido_Las_Matas_2.xlsx")

In [ ]:
guardarExcel(recorrido2,fname)

ValueError: Sheet 'Sheet1' already exists and if_sheet_exists is set to 'error'.

In [ ]:
recorrido_completo = df_df[['NTécnico', 'regular_days', 'periodo_inicio',
       'periodo_fin', 'company', 'product', 'comercial_product',
       'comercial_association_code_1', 'comercial_association_code_2',
       'distance_to_previous', 'type', 'code', 'lineCode',
       'lineCode_Complement',
       'comercial_value', 'order_value']].copy()

In [ ]:
renamed_columns = {
    "company":"Compañia",
    "product":"Producto",
    "comercial_product":"Producto comercial",
    "comercial_association_code_1":"Código asociación comercial 1",
    "comercie_1	comercial_association_code_2":"Código asociación comercial 2",
    "distance_to_previous":"Distancia al anterior (km)",
    "type":"Tipo",
    "code":"Código estación",
    "lineCode":"Código línea",
    "lineCode_Complement":"Código línea complemento",
    "comercial_value":"Valor comercial",
    "order_value":"Valor orden"}

In [ ]:
recorrido_completo.rename(columns=renamed_columns,inplace=True)

In [ ]:
recorrido_completo


,NTécnico,regular_days,periodo_inicio,periodo_fin,Compañia,Producto,Producto comercial,Código asociación comercial 1,comercial_association_code_2,Distancia al anterior (km),Tipo,Código estación,Código línea,Código línea complemento,Valor comercial,Valor orden
0,00040,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,2L63S14,0.0,comercial,71801,050,1,origen,1
1,00040,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,2L63S14,53.34,NaN,B4104,050,1,NaN,2
2,00040,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,2L63S14,28.6,NaN,04110,050,1,NaN,3
3,00040,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,2L63S14,107.54,NaN,04109,050,1,NaN,4
4,00040,1111111,2026-03-29,2026-12-12,RF,L,7,1L63S14,2L63S14,95.89,NaN,C0439,050,1,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321348,83510,0000001,2025-12-14,2026-05-10,RM,M,2,1RM,,23.91,NaN,73101,230,1,NaN,165
321349,83510,0000001,2025-12-14,2026-05-10,RM,M,2,1RM,,63.37,NaN,73102,230,1,NaN,166
321350,83510,0000001,2025-12-14,2026-05-10,RM,M,2,1RM,,47.0,NaN,C7310,230,1,NaN,167
321351,83510,0000001,2025-12-14,2026-05-10,RM,M,2,1RM,,44.02,NaN,71400,230,2,NaN,168


In [ ]:
recorrido_completo["Producto"].unique()

array(['L', 'R', 'C', 'I', 'U', 'X', 'N', 'M'], dtype=object)

In [ ]:
fecha_objetivo = pd.to_datetime("2026-04-14")


recorrido_completo = recorrido_completo[
    (recorrido_completo['periodo_inicio'].dt.date <= fecha_objetivo.date()) &
    (recorrido_completo['periodo_fin'].dt.date >= fecha_objetivo.date())]

In [ ]:
Cercania = recorrido_completo[recorrido_completo["Producto"] == "C"].copy()

In [ ]:
recorrido = pd.merge(
    origen[['NTécnico','comercial_association_code_1', 'comercial_association_code_2', 'code','product']],
    fin[['NTécnico','comercial_association_code_1', 'comercial_association_code_2', 'code']],
    on=['NTécnico','comercial_association_code_1', 'comercial_association_code_2'],
    suffixes=('_origen', '_final')
)

recorrido = recorrido.drop_duplicates(subset=['code_origen', 'code_final'])

In [ ]:
recorrido_cercania = recorrido[recorrido["product"] == "C"].copy()

In [ ]:
cercania_completo = Cercania[Cercania["NTécnico"].isin(recorrido_cercania["NTécnico"])].copy()

In [ ]:
cercania_completo = pd.merge(
    cercania_completo,
    recorrido_cercania[["NTécnico","code_origen","code_final"]],
    on="NTécnico",
    how="left"
).rename(columns={"code_origen":"Código_origen","code_final":"Código_destino"})

In [ ]:
estaciones.drop_duplicates(subset=["Código"],inplace=True)
regex = r'\s+(RAM|AV|RC)$'
estaciones['Nombre'] = estaciones['Nombre'].str.replace(regex, '', regex=True)

In [ ]:
cercanaia_completodo = cercania_completo.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código_origen',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_origen'}).drop(columns='Código')
cercanaia_completodo = cercanaia_completodo.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código_destino',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_destino'}).drop(columns='Código')
cercanaia_completodo = cercanaia_completodo.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código estación',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_estación'}).drop(columns='Código')


In [ ]:
cercanaia_completodo = cercanaia_completodo[["NTécnico","regular_days","periodo_inicio","periodo_fin","Compañia","Producto","Producto comercial","Código asociación comercial 1","comercial_association_code_2","Distancia al anterior (km)","Tipo","Código estación","Nombre_estación","Código línea","Código línea complemento",	"Valor comercial","Valor orden","Código_origen","Nombre_origen","Código_destino","Nombre_destino"]]

In [ ]:
media_distancia = recorrido_completo[recorrido_completo["Producto"] == "R"].copy()

In [ ]:
recorrido_media = recorrido[recorrido["product"] == "R"].copy()

In [ ]:
media_completa = media_distancia[media_distancia["NTécnico"].isin(recorrido_media["NTécnico"])].copy()

In [ ]:
media_completa = pd.merge(
    media_completa,
    recorrido_media[["NTécnico","code_origen","code_final"]],
    on="NTécnico",
    how="left"
).rename(columns={"code_origen":"Código_origen","code_final":"Código_destino"})
media_completa = media_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código_origen',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_origen'}).drop(columns='Código')
media_completa = media_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código_destino',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_destino'}).drop(columns='Código')
media_completa = media_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código estación',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_estación'}).drop(columns='Código')
media_completa = media_completa[["NTécnico","regular_days","periodo_inicio","periodo_fin","Compañia","Producto","Producto comercial","Código asociación comercial 1","comercial_association_code_2","Distancia al anterior (km)","Tipo","Código estación","Nombre_estación","Código línea","Código línea complemento",	"Valor comercial","Valor orden","Código_origen","Nombre_origen","Código_destino","Nombre_destino"]]

In [ ]:
larga_distancia = recorrido_completo[recorrido_completo["Producto"] == "L"].copy()
recorrido_larga = recorrido[recorrido["product"] == "L"].copy()
larga_completa = larga_distancia[larga_distancia["NTécnico"].isin(recorrido_larga["NTécnico"])].copy()


In [ ]:
larga_completa = pd.merge(
    larga_completa,
    recorrido_larga[["NTécnico","code_origen","code_final"]],
    on="NTécnico",
    how="left"
).rename(columns={"code_origen":"Código_origen","code_final":"Código_destino"})
larga_completa = larga_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código_origen',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_origen'}).drop(columns='Código')
larga_completa = larga_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código_destino',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_destino'}).drop(columns='Código')
larga_completa = larga_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código estación',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_estación'}).drop(columns='Código')
larga_completa = larga_completa[["NTécnico","regular_days","periodo_inicio","periodo_fin","Compañia","Producto","Producto comercial","Código asociación comercial 1","comercial_association_code_2","Distancia al anterior (km)","Tipo","Código estación","Nombre_estación","Código línea","Código línea complemento",	"Valor comercial","Valor orden","Código_origen","Nombre_origen","Código_destino","Nombre_destino"]]

In [ ]:
Mercancias  = recorrido_completo[recorrido_completo["Producto"] == "M"].copy()
recorrido_Mercancias = recorrido[recorrido["product"] == "M"].copy()
mercancias_completa = Mercancias[Mercancias["NTécnico"].isin(recorrido_Mercancias["NTécnico"])].copy()

In [ ]:
mercancias_completa = pd.merge(
    mercancias_completa,
    recorrido_Mercancias [["NTécnico","code_origen","code_final"]],
    on="NTécnico",
    how="left"
).rename(columns={"code_origen":"Código_origen","code_final":"Código_destino"})
mercancias_completa = mercancias_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código_origen',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_origen'}).drop(columns='Código')
mercancias_completa = mercancias_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código_destino',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_destino'}).drop(columns='Código')
mercancias_completa = mercancias_completa.merge(
    estaciones[['Código', 'Nombre']],
    left_on='Código estación',
    right_on='Código',
    how='left'
).rename(columns={'Nombre': 'Nombre_estación'}).drop(columns='Código')
mercancias_completa = mercancias_completa[["NTécnico","regular_days","periodo_inicio","periodo_fin","Compañia","Producto","Producto comercial","Código asociación comercial 1","comercial_association_code_2","Distancia al anterior (km)","Tipo","Código estación","Nombre_estación","Código línea","Código línea complemento",	"Valor comercial","Valor orden","Código_origen","Nombre_origen","Código_destino","Nombre_destino"]]

In [ ]:
cercanaia_completodo["Valor comercial"].fillna("Paso",inplace=True)
media_distancia["Valor comercial"].fillna("Paso",inplace=True)
larga_distancia["Valor comercial"].fillna("Paso",inplace=True)
mercancias_completa["Valor comercial"].fillna("Paso",inplace=True)

C:\Users\xiangzhou.zhang\AppData\Local\Temp\ipykernel_125152\272818771.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cercanaia_completodo["Valor comercial"].fillna("Paso",inplace=True)


In [ ]:
cercanaia_completodo.drop(columns=["regular_days","periodo_inicio","periodo_fin","Compañia","Producto comercial","Código línea complemento"],inplace=True)
media_completa.drop(columns=["regular_days","periodo_inicio","periodo_fin","Compañia","Producto comercial","Código línea complemento"],inplace=True)
larga_completa.drop(columns=["regular_days","periodo_inicio","periodo_fin","Compañia","Producto comercial","Código línea complemento"],inplace=True)
mercancias_completa.drop(columns=["regular_days","periodo_inicio","periodo_fin","Compañia","Producto comercial","Código línea complemento"],inplace=True)

In [ ]:
data = {
    "Recorrido_Cercanías": cercanaia_completodo,
    "Recorrido_Media_Distancia": media_completa,
    "Recorrido_Larga_Distancia": larga_completa
}

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\recorrido_completo.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

Guardando: Recorrido_Cercanías (1/3)
Guardando: Recorrido_Media_Distancia (2/3)
Guardando: Recorrido_Larga_Distancia (3/3)


<h1>Agrupación Comercial  </H1>

In [ ]:

fpath = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Listado de asociaciones Comerciales de Trenes BI-Xiang.xlsx")
L = pd.read_excel(fpath)
R = pd.read_excel(fpath, sheet_name="PRODUCTO MEDIA DISTANCIA")
M = pd.read_excel(fpath, sheet_name="PRODUCTO MERCANCÍAS")

In [ ]:
larga_completa["Código_comercial"] =larga_completa['Código asociación comercial 1'].str.extract(r'(L.*)')
media_completa['Código_comercial'] = media_completa['Código asociación comercial 1'].str.replace(r'^\d+', '', regex=True).str.replace(' ', '')
mercancias_completa['Código_comercial'] = mercancias_completa['Código asociación comercial 1'].str.replace(r'^\d+', '', regex=True).str.replace(' ', '')


In [ ]:
A = mercancias_completa[mercancias_completa['Código_comercial'] =="CM"]

In [ ]:
A.head(2)

,NTécnico,Producto,Código asociación comercial 1,comercial_association_code_2,Distancia al anterior (km),Tipo,Código estación,Nombre_estación,Código línea,Valor comercial,Valor orden,Código_origen,Nombre_origen,Código_destino,Nombre_destino,Código_comercial
0,41131,M,1CM,,0.0,comercial,71901,Barcelona Can Tunis,238,origen,1,71901,Barcelona Can Tunis,04313,LIMITE ADIF - LFPSA,CM
1,41131,M,1CM,,29.77,NaN,B7193,NaN,238,Paso,2,71901,Barcelona Can Tunis,04313,LIMITE ADIF - LFPSA,CM


In [ ]:
A.groupby(["Código_origen", "Código_destino"]).size().reset_index(name="count")

,Código_origen,Código_destino,count
0,04313,79011,22
1,11014,11102,4
2,11102,11014,4
3,11102,C1160,55
4,11200,71901,98
5,11600,16501,127
6,13408,17200,59
7,16501,13400,117
8,16501,65200,182
9,17200,13408,59


In [ ]:
L["Código_comercial"] = L["cod Asoc Nv1"] + L["cod Asoc Nv2"]
R["Código_comercial"] = R["cod Asoc Nv1"] + R["cod Asoc Nv2"]
M["Código_comercial"] = M["cod Asoc Nv1"]

In [ ]:

result = pd.merge(
    L,
    larga_completa[["Código_comercial","Código_origen","Nombre_origen","Código_destino","Nombre_destino"]],
    left_on=["Código_comercial"],
    right_on =["Código_comercial"],
    how = "left"
)
Media_filter = pd.merge(
    R,
    media_completa[["Código_comercial","Código_origen","Nombre_origen","Código_destino","Nombre_destino"]],
    left_on=["Código_comercial"],
    right_on =["Código_comercial"],
    how = "left"
)
Mercancias_filter = pd.merge(
    M,
    mercancias_completa[["Código_comercial","Código_origen","Nombre_origen","Código_destino","Nombre_destino"]],
    left_on=["Código_comercial"],
    right_on =["Código_comercial"],
    how = "left"
)


In [ ]:
result.drop(columns=["Nombre origen","Nombre destino","Ejemplo (num tec)","Unnamed: 0","Unnamed: 1"],inplace=True)
Media_filter.drop(columns=["Nombre origen","Nombre destino","Ejemplo (num tec)","Unnamed: 0","Unnamed: 1"],inplace=True)
Mercancias_filter.drop(columns=["Nombre origen","Nombre destino","Ejemplo (num tec)","Unnamed: 0","Unnamed: 1"],inplace=True)

In [ ]:
# result.drop(columns = ["Código_comercial"],inplace=True)

In [ ]:
result.drop_duplicates(inplace=True)
Media_filter.drop_duplicates(inplace=True)
Mercancias_filter.drop_duplicates(inplace=True)

In [ ]:
resultado_1 =  result.merge(
    larga_completa[['NTécnico', 'Código_comercial', 'Código_origen', 'Código_destino']],
    how='left',
    on=['Código_comercial', 'Código_origen', 'Código_destino']
)
media_resultado =  Media_filter.merge(
    media_completa[['NTécnico', 'Código_comercial', 'Código_origen', 'Código_destino']],
    how='left',
    on=['Código_comercial', 'Código_origen', 'Código_destino']
)
Mercancia_resultado =  Mercancias_filter.merge(
    mercancias_completa[['NTécnico', 'Código_comercial', 'Código_origen', 'Código_destino']],
    how='left',
    on=['Código_comercial', 'Código_origen', 'Código_destino']
)

In [ ]:
resultado_1.drop_duplicates(inplace=True)
media_resultado.drop_duplicates(inplace=True)
Mercancia_resultado.drop_duplicates(inplace=True)

In [ ]:
data = {
    "Larga_Distancia": resultado_1,
    "Media_Distancia": media_resultado,
    "Mercancias": Mercancia_resultado
}

In [ ]:
fname= Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Listado_asociaciones_comericales.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

Guardando: Larga_Distancia (1/3)
Guardando: Media_Distancia (2/3)
Guardando: Mercancias (3/3)


In [ ]:
no_comercial_ambos = circulación[(circulación["comercial_association_code_1"] == "") & (circulación["comercial_association_code_2"] == "")].copy()

In [ ]:
no_comercial_1 = circulación[(circulación["comercial_association_code_1"] == "") & (circulación["comercial_association_code_2"] != "")].copy()

In [ ]:
no_comercial_2 = circulación[(circulación["comercial_association_code_2"] == "") & (circulación["comercial_association_code_1"] != "")].copy()

In [ ]:
no_comercial_ambos.drop_duplicates(subset=["NTécnico"], inplace=True)
no_comercial_1.drop_duplicates(subset=["NTécnico"], inplace=True)
no_comercial_2.drop_duplicates(subset=["NTécnico"], inplace=True)

In [ ]:
no_comercial_ambos

,NTécnico,regular_days,product,comercial_product,comercial_association_code_1,comercial_association_code_2,rotation_code,rotation_calendar_code,rotation_regular_days,rotación
186855,36208,1111100,R,B,,,NaN,NaN,NaN,None
186926,36260,1111111,C,B,,,NaN,NaN,NaN,None
186984,36283,1111100,C,B,,,77708,LMXJV,1111100,77708
186994,36285,1111100,C,B,,,77980,LMXJV,1111100,77980
187014,36287,1111100,C,B,,,77710,LMXJV,1111100,77710
...,...,...,...,...,...,...,...,...,...,...
286330,79020,1111101,L,S,,,79005,D,0000001,None
286336,79021,1111100,L,S,,,NaN,NaN,NaN,None
286339,79022,1111100,L,S,,,79003,LMXJV,1111100,79003
286345,79023,1111100,L,S,,,NaN,NaN,NaN,None


In [ ]:
gct= getPlanificacionCirculacionesTecnicas("2026-02-20")

In [ ]:
gct  = gct[["NTécnico","Operador", "Tipo", "esComercial"]].copy()

In [ ]:
gct[gct["NTécnico"] == "36211"]

,NTécnico,Operador,Tipo,esComercial


In [ ]:
no_comercial_ambos = no_comercial_ambos.merge(
    gct,
    left_on="NTécnico",
    right_on="NTécnico", 
    how="left"
)


In [ ]:
no_comercial_1 = no_comercial_1.merge(
    gct,
    left_on="NTécnico",
    right_on="NTécnico", 
    how="left"
)

In [ ]:
no_comercial_2 = no_comercial_2.merge(
    gct,
    left_on="NTécnico",
    right_on="NTécnico", 
    how="left"
)

In [ ]:
data = {
    "Sin_comercial_ambos": no_comercial_ambos,
    "Sin_comercial_1": no_comercial_1,
    "Sin_comercial_2": no_comercial_2
}


In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\sin_comercial_2.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

Guardando: Sin_comercial_ambos (1/3)
Guardando: Sin_comercial_1 (2/3)
Guardando: Sin_comercial_2 (3/3)


<h1> Rolling stock </h1>

In [ ]:
df_df.columns

Index(['NTécnico', 'NComercial', 'rolling_inside_control_point',
       'regular_days', 'no_dates', 'periodo_inicio', 'periodo_fin', 'company',
       'product', 'comercial_product', 'comercial_association_code_1',
       'comercial_association_code_2', 'traction_provider', 'traction_number',
       'traction_code', 'weigth_value', 'length_value', 'security_rule_value',
       'danger_cargo_value', 'tle_value', 'distance_to_previous', 'type',
       'code', 'lineCode', 'lineCode_Complement', 'times_departure',
       'times_technical_departure', 'comercial_value', 'order_value',
       'times_arrival', 'speed_value', 'speed_type',
       'train_identificator_code', 'class_stop', 'rotation_code',
       'rotation_calendar_code', 'rotation_regular_days',
       'rotation_holiday_region'],
      dtype='object')

In [ ]:
rollings = df_df[df_df["rolling_inside_control_point"] == True].copy()

In [ ]:
rollings_filter = rollings[["NTécnico","company","product"]].copy()

In [ ]:
rollings_filter.drop_duplicates(inplace=True)

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Rolling_inside_journey.xlsx")

In [ ]:
guardarExcel(rollings_filter,fname)

In [ ]:
test

,NTécnico,regular_days,product,comercial_product,comercial_association_code_1,comercial_association_code_2,rotation_code,rotation_calendar_code,rotation_regular_days,rotación
0,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None
1,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None
2,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None
3,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None
4,00040,1111111,L,7,1L63S14,2L63S14,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...
322851,82911,0001100,M,1,1RM,,NaN,NaN,NaN,None
322852,82911,0001100,M,1,1RM,,NaN,NaN,NaN,None
322853,82911,0001100,M,1,1RM,,NaN,NaN,NaN,None
322854,82911,0001100,M,1,1RM,,NaN,NaN,NaN,None


In [ ]:
circulación = df_df[df_df["regular_days"].str[3] == "1"].copy()
circulación["rotación"] = np.where(
    circulación["rotation_regular_days"].str[3] == "1",
   circulación["rotation_code"],
    None
)
test= circulación[(circulación["rotation_regular_days"].str[3]=="1") | (circulación["rotation_regular_days"].isna())].copy()

In [ ]:
tess = test[["NTécnico","order_value","code","times_departure","times_technical_departure","departure","technical_departure"]].copy()

In [ ]:
df_filter = tess[tess["departure"] != tess["technical_departure"]].copy()

In [ ]:
df_filter

,NTécnico,order_value,code,times_departure,times_technical_departure,departure,technical_departure
22,00040,29,78400,0d 09:35:20,0d 09:37:20,0d 09:35:20,0d 09:37:20
101,00040,134,50500,0d 13:42:30,0d 13:51:40,0d 13:42:30,0d 13:51:40
118,00041,16,50500,0d 17:09:00,0d 17:20:00,0d 17:09:00,0d 17:20:00
146,00041,52,B6001,0d 19:12:40,0d 19:18:40,0d 19:12:40,0d 19:18:40
178,00041,97,04040,0d 20:54:20,0d 20:55:20,0d 20:54:20,0d 20:55:20
...,...,...,...,...,...,...,...
322795,82911,39,10400,0d 19:20:40,0d 19:31:20,0d 19:20:40,0d 19:31:20
322805,82911,50,10504,0d 21:02:20,0d 21:20:00,0d 21:02:20,0d 21:20:00
322807,82911,52,10600,0d 21:34:40,0d 21:42:00,0d 21:34:40,0d 21:42:00
322826,82911,72,11014,0d 23:36:30,0d 23:41:00,0d 23:36:30,0d 23:41:00


In [ ]:
df_filter.drop(columns=["departure","technical_departure"],inplace=True)

In [ ]:
estaciones = loadEstaciones()

In [ ]:
este = estaciones[estaciones["CTC"]=="EST1"].copy()

In [ ]:
df_filter[df_filter["NTécnico"] == "00040"]

,NTécnico,order_value,code,times_departure,times_technical_departure
22,00040,29,78400,0d 09:35:20,0d 09:37:20
101,00040,134,50500,0d 13:42:30,0d 13:51:40


In [ ]:
df_filter[df_filter["code"].isin(este["Código"])].head(20)

,NTécnico,order_value,code,times_departure,times_technical_departure
1690,00461,17,62002,0d 13:28:20,0d 13:29:10
1695,00461,22,62103,0d 13:53:10,0d 13:55:10
1699,00461,26,62109,0d 14:09:30,0d 14:21:30
1701,00461,28,60911,0d 14:40:10,0d 14:50:10
1709,00461,36,60907,0d 15:12:10,0d 15:14:40
1771,00462,35,65300,0d 13:32:00,0d 13:40:00
1822,00464,26,65300,0d 18:21:00,0d 18:24:00
1832,00464,36,65402,0d 19:24:00,0d 19:25:00
1837,00464,41,65421,0d 19:45:40,0d 19:50:00
1874,00467,20,64004,0d 15:57:20,0d 16:10:40


In [ ]:
este[este["Código"] == "62002"]

,Delegación,Catálogo,CTC,NombreCTC,Tecnólogo,Código,Nombre,Mnemónico,Mnemónico_comercial
1119,ESTE,EST1,EST1,ESTE 1: Valencia-FSL,DIMETRONIC,62002,Orihuela,OH,OH
